# Spark Structured Streaming: Kafka → Kafka and Kafka → MongoDB

The same logic as `jobs/stream_invoices.py`, cell by cell. Run it with `make notebook` (http://localhost:8888).

**Stop the `spark-stream` service first** (`docker compose stop spark-stream`) if you want to see the notebook process the data from the beginning; the notebook uses its own checkpoints (`/checkpoints/notebook`), so it never interferes with the job's progress.

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

spark = (SparkSession.builder.appName('invoices-notebook')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
KAFKA = os.environ['KAFKA_BOOTSTRAP']
CHECKPOINTS = os.environ['CHECKPOINT_DIR']
spark

## 1. Read the stream
A Kafka record has `key`, `value` (bytes), `topic`, `partition`, `offset`, `timestamp`. The value is the JSON the API produced.

In [ ]:
raw = (spark.readStream.format('kafka')
       .option('kafka.bootstrap.servers', KAFKA)
       .option('subscribe', 'ingestion-topic')
       .option('startingOffsets', 'earliest')
       .option('maxOffsetsPerTrigger', 5000)
       .load())
raw.printSchema()

## 2. Parse JSON with an explicit schema
Never infer schemas on a stream: an explicit schema is the contract, and malformed records become NULLs you can filter (or route to a dead-letter topic).

In [ ]:
schema = StructType([
    StructField('InvoiceNo', StringType(), False), StructField('StockCode', StringType(), False),
    StructField('Description', StringType()), StructField('Quantity', IntegerType()),
    StructField('InvoiceDate', TimestampType()), StructField('UnitPrice', DoubleType()),
    StructField('CustomerID', StringType()), StructField('Country', StringType()),
])
items = (raw.select(F.from_json(F.col('value').cast('string'), schema).alias('item'))
            .select('item.*')
            .where(F.col('InvoiceNo').isNotNull()))

## 3. Peek at the stream (memory sink, for exploration only)

In [ ]:
peek = items.writeStream.format('memory').queryName('items_peek').trigger(availableNow=True).start()
peek.awaitTermination()
spark.sql('select InvoiceNo, count(*) items, round(sum(Quantity * UnitPrice), 2) total from items_peek group by 1 order by 1 limit 10').show()

## 4. Kafka → Kafka (`spark-output`)
Add a computed column and write JSON back to Kafka, keyed by invoice. `availableNow` processes what is there and stops: handy in a notebook.

In [ ]:
to_kafka = (items.withColumn('LineTotal', F.round(F.col('Quantity') * F.col('UnitPrice'), 2))
    .select(F.col('InvoiceNo').alias('key'), F.to_json(F.struct('*')).alias('value'))
    .writeStream.format('kafka')
    .option('kafka.bootstrap.servers', KAFKA)
    .option('topic', 'spark-output')
    .option('checkpointLocation', f'{CHECKPOINTS}/spark-output')
    .trigger(availableNow=True).start())
to_kafka.awaitTermination()
to_kafka.lastProgress

## 5. Kafka → MongoDB (nested documents)
`foreachBatch` gives a normal DataFrame per micro-batch; `build_updates` groups items into one document per invoice and upserts idempotently (`$addToSet`).

In [ ]:
from stream_invoices import write_to_mongo

to_mongo = (items.writeStream.foreachBatch(write_to_mongo)
    .option('checkpointLocation', f'{CHECKPOINTS}/mongo')
    .trigger(availableNow=True).start())
to_mongo.awaitTermination()

In [ ]:
from pymongo import MongoClient
invoices = MongoClient(os.environ['MONGO_URI']).get_default_database()['invoices']
invoices.count_documents({}), invoices.find_one({}, {'_id': 0})